# Open Notebook in Colab 

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/childmindresearch/llm_tracker/blob/main/tutorials/discovery_tutorial.ipynb)


Please cite this if you use llm_tracker: <br>
Low, D.M., Mair, P., Nock, M., & Ghosh, S. (2026). Text Psychometrics: Assessing Psychological Constructs in Text Using Natural Language Processing. PLOS Digital Health.

# Discover constructs and build a codebook

Use this tutorial to explore psychological constructs in your texts before you
have a codebook. You choose a theoretical framework, such as cognitive behavioral
therapy. A large language model (LLM) then suggests constructs supported by the
texts and organizes related ideas into groups. You review these suggestions,
make changes, and create a codebook for coding other documents.

Work through the notebook from top to bottom. Each code cell is a block of
instructions with a Run button. The setup cell below prepares the software.
If you update the software while the notebook is open, restart the Colab session
or Jupyter kernel (the process that runs the code), then start again from setup.

Finding constructs and grouping related ideas send requests to the LLM service
and may use paid credits. Reviewing groups, setting aside constructs, preparing
tables, and creating the codebook do not send additional requests. An optional
later step asks the LLM to review group names and definitions. The notebook does
not save your API key, the access key for the LLM service.


# Setup (first time only)

Run the next cell to download LLM Tracker and the practice files, and install
the software needed for this tutorial. You do not need to download the project
yourself beforehand. In Google Colab, you can start with this notebook and an
internet connection.

You can also use Jupyter on your own computer with Python 3.11–3.13 installed.
Setup may take a few minutes. If asked to restart the Colab session or Jupyter
kernel, do so and run this setup cell again before continuing. Setup does not
send requests to an LLM or spend API credits.

In [1]:
from pathlib import Path
import subprocess
import sys
import tempfile
from urllib.request import urlretrieve
from zipfile import ZipFile
import os 
from getpass import getpass

# Find an existing copy, or download LLM Tracker and the sample data.
project_dir = Path.cwd()
if not (project_dir / "src" / "llm_tracker").is_dir():
    project_dir = project_dir.parent

if not (project_dir / "src" / "llm_tracker").is_dir():
    project_dir = Path.cwd() / "llm_tracker-main"
    if not (project_dir / "src" / "llm_tracker").is_dir():
        print("Downloading LLM Tracker...")
        with tempfile.TemporaryDirectory() as folder:
            archive = Path(folder) / "llm_tracker.zip"
            urlretrieve(
                "https://codeload.github.com/childmindresearch/llm_tracker/zip/refs/heads/main",
                archive,
            )
            with ZipFile(archive) as downloaded:
                downloaded.extractall(project_dir.parent)

# Allow NumPy 2 for Python 3.13 compatibility.
pyproject = project_dir / "pyproject.toml"
settings = pyproject.read_text()
settings = settings.replace('numpy = "^1.26"', 'numpy = ">=1.26,<3"')
pyproject.write_text(settings)

# Install the package; stop if installation fails.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--only-binary=numpy,pandas,scikit-learn",
        "--timeout", "30", "--retries", "1",
        "-e", str(project_dir), "openpyxl", "xlrd",
    ],
    check=True,
)

sys.path.insert(0, str(project_dir / "src"))

# Make the bundled sample-data paths work.
%cd "{project_dir}"

import llm_tracker.discovery
print("Discovery setup complete.")


Obtaining file:///Users/freymon.perez/Projects/GitHub/llm_tracker
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for llm-tracker (pyproject.toml): started
  Building editable for llm-tracker (pyproject.toml): finished with status 'done'
  Created wheel for llm-tracker: filename=llm_tracker-0.1.0b1-py3-none-any.whl size=10615 sha256=ca20f5fa06aadcd34b6e1d646c5092e2f4e07a421fc3faf1aec30da96b843a1f
  Stored in directory: /private/var/folders/09/drdv5jw504142vpxfq711jf815q1qc/T/pip-ephem-wheel-cache-kx6q0vmx/wheels/9d/32/98/c6659


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
/Users/freymon.perez/Projects/GitHub/llm_tracker/src/llm_tracker/models.py:40: UserWarning: Field name "construct" in "ConstructInstance" shadows an attribute in parent "BaseModel"
  class ConstructInstance(BaseModel):


Discovery setup complete.


## 1. Choose your data and settings

Choose how to provide your texts by changing `input_source` in the next cell.
A path is the address of a file or folder.

| Choose | What happens |
|---|---|
| `"demo"` | Creates six short practice texts. No file upload or Google Drive connection is needed. |
| `"path"` | Opens a file or folder at the address in `input_path`. The example uses the included Reddit sample. For your own file in the project's `data/` folder, use `input_path = Path("data/my_data.csv")`. If running Jupyter on your computer, you can also give the full address of a file there. |
| `"upload"` | **Colab only:** lets you choose one file from your computer. Colab keeps a temporary copy; it disappears when the Colab session's storage is cleared. |
| `"drive"` | **Colab only:** asks permission to connect to Google Drive, then opens the file or folder at `drive_path`. For a file in a Research folder in My Drive, use an address such as `/content/drive/MyDrive/Research/my_data.csv`. |

For your own data, set `text_column` to the heading of the column containing
your texts. If you have a column of document identifiers, enter its heading as
`id_column`; otherwise leave `id_column=None`. The demo fills these in for you.

An address such as `data/my_data.csv` starts from the project folder prepared
during setup. In Colab, files on your computer are not available automatically.
Use Colab's Files sidebar to place a file in `data/`, choose `"upload"`, or
choose `"drive"` to read it from Google Drive.

- **Text files (TXT):** each file is one document. Set `text_column=None` and
  `id_column=None`.
- **Spreadsheets (CSV, XLSX, or XLS):** set `text_column` so that each row is
  treated as a separate document. Without it, the entire spreadsheet is treated
  as one document.
- **Folders:** files directly inside the folder are read, but files inside
  further subfolders are not. All spreadsheets must use the same column settings.

Results are saved separately from your original data. `output_root` is the
main results folder. When using `"drive"`, you can set it to an address inside
`/content/drive/MyDrive/` to keep results in Google Drive. Otherwise, download
your results before Colab clears the session's storage.

`model_name` selects the LLM. The default, `google/gemini-flash-latest`, uses
Gemini Flash latest through OpenRouter, the service connecting this notebook
to the model. You can enter another model's OpenRouter name instead.
The key-entry cell uses `OPENROUTER_API_KEY` if you have already made your
access key available under that name. Otherwise, it asks you to enter the key
without showing its characters on screen.


In [31]:
from pathlib import Path

input_source = "path"  # Choose "demo", "path", "upload", or "drive".

framework = "Modern clinical psychology constructs (CBT, DBT, ACT, etc.)"  # Other options: "Cognitive behavioral therapy" "Political beliefs" "Buddhism"

input_path = Path("sample_data/reddit_autism_anxiety_depression.csv")  # For "path".
drive_path = Path("/content/drive/MyDrive/YOUR_FILE.csv")  # For "drive".
output_root = Path("discovery_outputs")

text_column = "post"  # For your own tabular file, replace with its text column.
id_column = None  # Uses the row number; or set your document ID column name.

provider = "openrouter"
model_name = "~google/gemini-flash-latest"
max_retries = 2

fuzzy_threshold = 0.9  # None disables fuzzy spelling matching.
seed = 0
run_reevaluation = False  # True makes an additional LLM call after filtering.
n_codebook_examples = 3


In [ ]:
api_key = os.environ.get("OPENROUTER_API_KEY", "").strip() or getpass("OpenRouter API key: ").strip()
if not api_key:
    raise ValueError("Enter your OpenRouter API key.")


In [16]:
import json
from copy import deepcopy

import pandas as pd
from IPython.display import display

import llm_tracker.discovery as discovery_module
from llm_tracker.config import AnalyzerConfig
from llm_tracker.discovery import (
    DISCOVERY_PROMPT,
    MERGE_PROMPT,
    REEVALUATE_PROMPT,
    LLMTrackerDiscoverer,
    apply_merge_edits,
    build_codebook,
    build_discovery_tables,
    codebook_to_frame,
    collapse_variants,
    filter_merged_dataframe,
    filter_merged_constructs,
    format_merged_constructs,
    frame_to_codebook,
    merge_constructs,
    reevaluate_merged_constructs,
    save_codebook,
)

print("Discovery module:", discovery_module.__file__)
print("Working directory:", Path.cwd())


Discovery module: /Users/freymon.perez/Projects/GitHub/llm_tracker/src/llm_tracker/discovery.py
Working directory: /Users/freymon.perez/Projects/GitHub/llm_tracker


Run the next cell to open the data you selected. If uploading, choose **one
file**. If using Google Drive, approve access and check that `drive_path` gives
the address of your file or folder. Colab cannot open a file directly from an
address on your computer, such as `/Users/...` on a Mac; upload it or use Drive.

Practice texts are created only when you choose `input_source="demo"`. If the
practice file already exists, it is kept rather than replaced. Check the file
address printed below to confirm which texts will be sent to the LLM later.


In [17]:
if input_source not in {"demo", "path", "upload", "drive"}:
    raise ValueError('Choose input_source = "demo", "path", "upload", or "drive".')

if input_source == "upload":
    try:
        from google.colab import files
    except ImportError as error:
        raise RuntimeError('Upload is for Colab. On your computer, choose "path" and set input_path.') from error
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Choose exactly one file, then rerun this cell.")
    input_path = Path(next(iter(uploaded))).resolve()
elif input_source == "drive":
    try:
        from google.colab import drive
    except ImportError as error:
        raise RuntimeError('Drive mounting is for Colab. On your computer, choose "path" and use your local Drive folder.') from error
    drive.mount("/content/drive")
    input_path = drive_path.expanduser().resolve()
    if not input_path.is_relative_to(Path("/content/drive")):
        raise ValueError("Set drive_path to a file or folder under /content/drive/MyDrive/.")

if input_source == "demo":
    input_path = Path("data/discovery_demo.csv")
    text_column, id_column = "text", "doc_id"
    input_path.parent.mkdir(parents=True, exist_ok=True)
    if not input_path.exists():
        demo = pd.DataFrame([
            {"doc_id": "demo_1", "text": "If I make one mistake at work, everyone will think I am incompetent."},
            {"doc_id": "demo_2", "text": "I assume my friend is angry whenever she takes a while to reply."},
            {"doc_id": "demo_3", "text": "I felt anxious about the meeting, so I avoided opening the invitation."},
            {"doc_id": "demo_4", "text": "I wrote down the evidence for and against my fear, which helped me consider another explanation."},
            {"doc_id": "demo_5", "text": "I took one small step toward the task instead of putting it off again."},
            {"doc_id": "demo_6", "text": "The delivery contained three blue folders and a box of paper clips."},
        ])
        demo.to_csv(input_path, index=False)
    display(pd.read_csv(input_path))

input_path = input_path.expanduser().resolve()
if not input_path.exists():
    raise FileNotFoundError(f"Update input_path; no input exists at {input_path.resolve()}")

print("Input selected:", input_path)
output_root = output_root.expanduser().resolve()
print("Outputs will save to:", output_root)

discovery_dir = output_root / "discovery"
merge_dir = output_root / "merged"
edits_dir = output_root / "edited"
filtered_dir = output_root / "filtered"
reevaluation_dir = output_root / "reevaluated"
tables_dir = output_root / "final_tables"
codebook_dir = output_root / "codebooks"
output_root.mkdir(parents=True, exist_ok=True)


Input selected: /Users/freymon.perez/Projects/GitHub/llm_tracker/sample_data/reddit_autism_anxiety_depression.csv
Outputs will save to: /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs


## 2. Review the instructions for finding constructs

A prompt is the written instruction sent to the LLM. The next cell shows the
instructions for identifying constructs in your texts. Reading or displaying
these instructions does not send anything to the LLM service.

You can keep the standard instructions. If you edit them, leave `{framework}`
and `{document}` in place: the notebook fills these in with your chosen
framework and each document's text. If adding other curly brackets, write them
as `{{` and `}}` so they are not mistaken for places to insert information.
The short example below shows how to change the instructions and then restores
the standard version before you continue.


In [18]:
config_kwargs = {
    "provider": provider,
    "max_retries": max_retries,
    "temperature": 0.0,
}
if model_name is not None:
    config_kwargs["model_name"] = model_name
if api_key is not None:
    config_kwargs["api_key"] = str(api_key)

config = AnalyzerConfig(**config_kwargs)
discoverer = LLMTrackerDiscoverer(config=config)
print("Model:", config.model_name)
print(discoverer.discovery_prompt)


Model: google/gemini-3-flash-preview
You are an expert qualitative researcher analyzing a document through the lens of a specific theoretical framework.

Framework: {framework}

Read the document below and identify the constructs from this framework that are present in the text. Use your own judgement about what counts as a construct of this framework; you are not working from a fixed list.

For each construct you find, provide:
- "construct": a short name for the construct
- "description": one sentence describing the construct as it appears here
- "quotes": a list of verbatim quotes from the document that evidence it

Only report constructs genuinely grounded in the text. If none are present, return an empty list.
Spell out construct names and descriptions in full. Do not use acronyms such as
CBT; write cognitive behavioral therapy instead.

Respond ONLY with a JSON object of the form:
{{"constructs": [{{"construct": "...", "description": "...", "quotes": ["..."]}}]}}

Document:
{docu

In [19]:
# Replace the prompt with a simple message so we can see that it changed.
discoverer.discovery_prompt = "We changed the prompt completely!"
print(discoverer.discovery_prompt)

# Change it back to the default prompt before running discovery.
discoverer.discovery_prompt = DISCOVERY_PROMPT
assert discoverer.discovery_prompt == DISCOVERY_PROMPT


We changed the prompt completely!


## 3. Discover constructs in the documents

This step sends your texts to the LLM, usually one request per document.
If a request fails, the notebook may try again, up to the number set in
`max_retries`. Running this cell again sends all the documents again and may
use more credits. Later steps reuse the results table rather than repeating
this analysis.

The results show one row for each supporting quote. If the model identifies a
construct but provides no quote, the row is kept with an empty quote field.
Documents analyzed successfully still count in the total used to calculate
percentages, even if the model finds no constructs in them. Documents that
could not be analyzed are excluded. If there are errors, the notebook stops
so you can review them before interpreting the findings.


In [20]:
discoveries, discovery_metadata, discovery_errors = discoverer.discover(
    path=input_path,
    framework=framework,
    text_column=text_column,
    id_column=id_column,
    output_dir=discovery_dir,
    save_format="both",
)

print("Successfully processed documents:", discoveries.attrs["total_documents"])
print("Discovery rows:", len(discoveries))
print("Failures:", len(discovery_errors))
display(discoveries)

if discovery_errors:
    display(pd.DataFrame([error.model_dump() for error in discovery_errors]))
    raise RuntimeError("Review the saved discovery errors before continuing. Rerunning discovery processes every document again.")
if discoveries.empty:
    raise RuntimeError("No constructs were discovered. Review the inputs, framework, and prompt before merging.")


Discovering across 60 document(s) from 1 file(s).

Discovering [1/60]: reddit_autism_anxiety_depression.csv::0
  OK: 5 construct(s), 10 quote(s)
Discovering [2/60]: reddit_autism_anxiety_depression.csv::1
  OK: 6 construct(s), 8 quote(s)
Discovering [3/60]: reddit_autism_anxiety_depression.csv::2
  OK: 4 construct(s), 7 quote(s)
Discovering [4/60]: reddit_autism_anxiety_depression.csv::3
  OK: 5 construct(s), 9 quote(s)
Discovering [5/60]: reddit_autism_anxiety_depression.csv::4
  OK: 4 construct(s), 7 quote(s)
Discovering [6/60]: reddit_autism_anxiety_depression.csv::5
  OK: 4 construct(s), 4 quote(s)
Discovering [7/60]: reddit_autism_anxiety_depression.csv::6
  OK: 4 construct(s), 7 quote(s)
Discovering [8/60]: reddit_autism_anxiety_depression.csv::7
  OK: 4 construct(s), 4 quote(s)
Discovering [9/60]: reddit_autism_anxiety_depression.csv::8
  OK: 5 construct(s), 10 quote(s)
Discovering [10/60]: reddit_autism_anxiety_depression.csv::9
  OK: 4 construct(s), 4 quote(s)
Discovering [11/

,filename,doc_id,construct,description,quote
0,reddit_autism_anxiety_depression.csv,0,social anxiety,The individual experiences significant distres...,I know my anxiety has made it hard for me to g...
1,reddit_autism_anxiety_depression.csv,0,social anxiety,The individual experiences significant distres...,When i talk to strangers it's one they I'll pr...
2,reddit_autism_anxiety_depression.csv,0,negative self-schema,The individual holds a core belief that they a...,I'm not a interesting person
3,reddit_autism_anxiety_depression.csv,0,negative self-schema,The individual holds a core belief that they a...,"i feel like saying watching tv, reading, fishi..."
4,reddit_autism_anxiety_depression.csv,0,avoidance behavior,The individual describes a history of social w...,anxiety can make it hard for people to go out ...
...,...,...,...,...,...
444,reddit_autism_anxiety_depression.csv,59,social isolation,The individual experiences a profound lack of ...,dealing with loneliness
445,reddit_autism_anxiety_depression.csv,59,social isolation,The individual experiences a profound lack of ...,when you have 0 friends and been forever alone...
446,reddit_autism_anxiety_depression.csv,59,social isolation,The individual experiences a profound lack of ...,with no close family members
447,reddit_autism_anxiety_depression.csv,59,anxiety,The individual identifies persistent feelings ...,Ive been dealing with loneliness and anxiety f...


## 4. Check for spelling differences in construct names

Slightly different names can refer to the same idea. This step checks for
differences such as "cognitive behavioral therapy" and "cognitive behavior
therapy" before the LLM groups constructs by meaning in the next step.

Names are made lowercase, and differences in spaces and separators such as
underscores are removed. The optional spelling check also combines names with
similar spellings. It compares spelling, not meaning. Set `fuzzy_threshold=None`
to turn off that extra check while keeping the basic cleanup. This step does
not send requests to the LLM.

In the table below, `count` is the number of different documents containing
each construct. A document is identified by its source file and document ID
together, so the same ID in two files refers to two documents. `instance_count`
counts supporting quotes, including repeated quote text; empty quotes are not
counted. Later tables use the name `document_count` instead of `count`.


In [21]:
normalized = collapse_variants(discoveries, fuzzy_threshold=fuzzy_threshold)
display(normalized[["construct", "count", "instance_count", "variants"]])
display(pd.DataFrame(normalized.attrs.get("name_matches", [])))


,construct,count,instance_count,variants
0,emotional dysregulation,17,33,"[Emotional Dysregulation, Emotional dysregulat..."
1,catastrophizing,12,18,"[Catastrophizing, catastrophizing]"
2,cognitive appraisal,8,12,"[cognitive appraisal, cognitive reappraisal]"
3,social anxiety,8,14,"[Social Anxiety, Social anxiety, social anxiety]"
4,experiential avoidance,7,8,[experiential avoidance]
...,...,...,...,...
138,emotional overwhelm,1,2,[Emotional Overwhelm]
139,emotional numbing,1,3,[emotional numbing]
140,emotional masking,1,2,[Emotional Masking]
141,emotional distress,1,1,[emotional distress]


,original_name,construct_found
0,Agoraphobia,agoraphobia
1,Alexithymia,alexithymia
2,Anhedonia,anhedonia
3,Anticipatory Anxiety,anticipatory anxiety
4,Anxiety Sensitivity,anxiety sensitivity
...,...,...
183,subjective experience of illness,subjective experience of illness
184,suicidal ideation,suicidal ideation
185,suicide planning,suicide planning
186,therapeutic alliance,therapeutic alliance


## 5. Group constructs by meaning

The LLM now looks for constructs that describe related ideas. It puts
them into groups and gives each group a name and definition. For example, it might
place "self hate" and "low self esteem" under "Negative self-perception".

This step is called merging. The example uses the standard grouping instructions,
`MERGE_PROMPT`. You can review or edit them, but keep `{framework}`, `{constructs}`,
and `{existing_note}` in place so the notebook can insert the information needed.

You can leave the settings below as they are for a first run. This table explains
the options used by `merge_constructs()` if you want to adjust them later:

| Setting | What it means |
|---|---|
| `df` | The results table to organize. Use the full results from the discovery step. |
| `config` | The model and connection settings already chosen above. |
| `framework` | The perspective the model uses to decide which ideas belong together. |
| `merge_prompt` | The written instructions for grouping the constructs. |
| `fuzzy_threshold` | How similar two spellings must be before treating them as one name. A higher number is stricter; `None` turns spelling matching off. |
| `seed` | Reusing this number puts the construct names in the same shuffled order. The LLM may still give different answers. |
| `batch` | `False` sends the full list in one request. `True` sends smaller portions, one after another. |
| `batch_size` | The number of construct names in each portion when using smaller requests. The default is 150. |
| `max_prompt_chars` | The maximum number of characters allowed in a single grouping request. An oversized request is stopped before it is sent. |
| `existing` | Optional earlier grouping to show the model; this walkthrough starts a new grouping. |
| `total_documents` | The number of successfully analyzed documents, including those with no constructs found. This is filled in automatically; enter it yourself only if reopening saved results that no longer include it. |
| `output_dir` | Where the results should be saved. |
| `save_format` | `"both"` saves a CSV file you can open in spreadsheet software and a JSON file that keeps grouped information, such as lists of quotes, together for later use. |


In [22]:
merged, merge_table = merge_constructs(
    df=discoveries,
    config=config,
    framework=framework,
    merge_prompt=MERGE_PROMPT,
    fuzzy_threshold=fuzzy_threshold,
    seed=seed,
    batch=False,
    max_prompt_chars=60000,
    output_dir=merge_dir,
    save_format="both",
)

display(format_merged_constructs(merge_table, n_quotes=3, seed=seed))


Collapsed 188 discovered name(s) into 143 distinct construct(s).
  Note: 36 construct(s) were not placed by the merge and are kept as their own canonical constructs (status='unplaced'): ['generalized anxiety', 'interpersonal conflict', 'nostalgia', 'subjective experience of illness', 'substance use as coping', 'social comparison', 'therapeutic alliance', 'treatment non adherence', 'pharmacotherapy', 'perceived loss of control']...
Saved merge output to /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs/merged


,merged_construct,definition,status,construct_found,document_count,instance_count,document_frequency_percent,total_documents,prototypicality,quotes,document_frequency
0,Avoidance Behaviors,Overt or covert actions taken to prevent or es...,merged,avoidance behavior,4,6,6.666667,60,1.00,I’m sitting alone in the cafeteria barely pick...,6.7% (4/60 documents)
1,Avoidance Behaviors,Overt or covert actions taken to prevent or es...,merged,behavioral avoidance,1,2,1.666667,60,0.95,He also refuses to even try to use the potty.;...,1.7% (1/60 documents)
2,Avoidance Behaviors,Overt or covert actions taken to prevent or es...,merged,experiential avoidance,7,8,11.666667,60,0.90,What do you do as a means of pushing yourself ...,11.7% (7/60 documents)
3,Avoidance Behaviors,Overt or covert actions taken to prevent or es...,merged,safety seeking behavior,1,2,1.666667,60,0.80,It always makes me feel a little better knowin...,1.7% (1/60 documents)
4,Avoidance Behaviors,Overt or covert actions taken to prevent or es...,merged,maladaptive coping mechanisms,3,4,5.000000,60,0.70,I get drunk usually every other day now and cr...,5.0% (3/60 documents)
...,...,...,...,...,...,...,...,...,...,...,...
138,stereotype threat mitigation,The active avoidance of reductive and harmful ...,unplaced,stereotype threat mitigation,1,2,1.666667,60,NaN,actually good representation as opposed to har...,1.7% (1/60 documents)
139,subjective experience of illness,The personal and internal reaction to receivin...,unplaced,subjective experience of illness,1,1,1.666667,60,NaN,What was it like to be diagnosed with autism,1.7% (1/60 documents)
140,substance use as coping,The individual utilizes chemical substances to...,unplaced,substance use as coping,1,1,1.666667,60,NaN,I’m sober for first day in a while like pretty...,1.7% (1/60 documents)
141,therapeutic alliance,The individual is engaging in a collaborative ...,unplaced,therapeutic alliance,1,1,1.666667,60,NaN,My therapist thinks I might be autistic but th...,1.7% (1/60 documents)


Each row shows one **discovered construct** and the **group** it belongs to.
The table calls these merged groups. A group can appear on several rows because
it contains several discovered constructs.

| Column | In plain language |
|---|---|
| `merged_construct` | The name of the larger group, such as "Negative self-perception". |
| `definition` | What that group means. |
| `construct_found` | The name originally found in the documents, such as "self hate", after spelling cleanup. |
| `document_count` | How many different documents contained this discovered construct. Finding it three times in one document still counts as one document. |
| `instance_count` | How many supporting quotes were found, excluding empty quotes. Repeated quote text is counted each time it was found, even across different documents. |
| `document_frequency` | How common it was: `1.7% (1/60 documents)` means one of the 60 successfully processed documents contained it. |
| `document_frequency_percent` | The same percentage as a number, useful for calculations. |
| `total_documents` | All successfully processed documents, including documents where nothing was found. Failed documents are excluded. |
| `prototypicality` | How well this discovered construct represents its merged group. A score near **1** means a strong, typical example; a score near **0** means a less typical member. This is **not** the confidence that a quote was detected correctly, or how often it appeared. |
| `quotes` | Supporting text. The table shown above displays up to three quotes per row. The full results keep all quotes, with identical text listed only once per row. |
| `status` | `merged`: assigned to a group. `standalone`: kept on its own. `unplaced`: not assigned by the model, but still kept in the results so it is not lost. |

An empty prototypicality score means no score is available; it does not mean zero.
Later steps use `merge_table`, the full results table. The shortened display is
only a reading aid; it does not replace or remove information from that table.
Here, `batch=True` means sending smaller requests one after another, not using
a provider's separate service for submitting jobs to be processed later.


### Choose minimum document counts and group-fit scores

You may want to set aside constructs found in only a few documents or judged
less typical of their group. The next cell separates them into two tables:

- **Kept:** constructs found in at least **2 different documents** and with a
  **prototypicality score of at least 0.3**. This score describes how well an
  idea fits its group, from 0 (less typical) to 1 (very typical).
- **Removed:** everything that did not meet one or both minimums. A construct
  without a score is also set aside while the score minimum is turned on.

A construct must meet both minimums to be kept; values exactly at the minimums
qualify. The document minimum counts different documents, not percentages or
quotes. Each discovered construct is checked separately, rather than the group
as a whole. Change the two numbers below to choose your minimums. Set either to
`None` to turn off that check.

Both tables are saved in your main results folder, `output_root`, as CSV files
for spreadsheets and JSON files for later use in the notebook. Their names start
with `filtered_merge_table_` and `removed_merge_table_`.
Nothing is deleted from the original merged table. You can inspect the removed
rows and rerun this cell with different minimums without sending another request
to the LLM. Later steps use the kept rows. With a small practice dataset, these
minimums may remove everything; lower them or turn them off to continue.


In [23]:
min_document_count = 2
min_prototypicality = 0.3

threshold_table, removed_table = filter_merged_dataframe(
    table=merge_table,
    min_document_count=min_document_count,
    min_prototypicality=min_prototypicality,
    output_dir=output_root,
    save_format="both",
)
print("Kept constructs:", len(threshold_table))
display(format_merged_constructs(threshold_table, n_quotes=3, seed=seed))
print("Removed constructs:", len(removed_table))
display(format_merged_constructs(removed_table, n_quotes=3, seed=seed))
print("Both tables saved to:", output_root)

if threshold_table.empty:
    raise RuntimeError("No constructs met both minimums. Lower the values above or set them to None, then rerun this cell.")


Saved retained and removed merge tables to /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs
Kept constructs: 34


,merged_construct,definition,status,construct_found,document_count,instance_count,document_frequency_percent,total_documents,prototypicality,quotes,document_frequency
0,Avoidance Behaviors,Overt or covert actions taken to prevent or es...,merged,avoidance behavior,4,6,6.666667,60,1.00,I’m sitting alone in the cafeteria barely pick...,6.7% (4/60 documents)
2,Avoidance Behaviors,Overt or covert actions taken to prevent or es...,merged,experiential avoidance,7,8,11.666667,60,0.90,What do you do as a means of pushing yourself ...,11.7% (7/60 documents)
4,Avoidance Behaviors,Overt or covert actions taken to prevent or es...,merged,maladaptive coping mechanisms,3,4,5.000000,60,0.70,I get drunk usually every other day now and cr...,5.0% (3/60 documents)
6,Behavioral Change Interventions,Active therapeutic strategies aimed at modifyi...,merged,behavioral activation,4,6,6.666667,60,0.90,Over the years I've attempted lots of exercise...,6.7% (4/60 documents)
14,Cognitive Distortions,Biased or exaggerated thought patterns that re...,merged,cognitive distortions,6,9,10.000000,60,1.00,what's wrong with me that everyone goes out of...,10.0% (6/60 documents)
15,Cognitive Distortions,Biased or exaggerated thought patterns that re...,merged,catastrophizing,12,18,20.000000,60,0.90,now I’m freaking out that it has important inf...,20.0% (12/60 documents)
18,Cognitive Distortions,Biased or exaggerated thought patterns that re...,merged,cognitive distortion of labeling,4,4,6.666667,60,0.80,thinking about all of the ‘bad’ things (not re...,6.7% (4/60 documents)
19,Cognitive Flexibility and Metacognition,The ability to adapt mental frameworks to new ...,merged,cognitive flexibility,2,4,3.333333,60,1.00,I'm trying my best to have an open mind but it...,3.3% (2/60 documents)
21,Cognitive Flexibility and Metacognition,The ability to adapt mental frameworks to new ...,merged,cognitive appraisal,8,12,13.333333,60,0.80,My roommate is really annoying;\nNow that's no...,13.3% (8/60 documents)
25,Depressive Symptomatology,"A cluster of symptoms including low mood, hope...",merged,depressive affect,3,8,5.000000,60,0.95,I feel so low;\nI feel so empty.;\nit fills me...,5.0% (3/60 documents)


Removed constructs: 109


,merged_construct,definition,status,construct_found,document_count,instance_count,document_frequency_percent,total_documents,prototypicality,quotes,document_frequency
1,Avoidance Behaviors,Overt or covert actions taken to prevent or es...,merged,behavioral avoidance,1,2,1.666667,60,0.95,He also refuses to even try to use the potty.;...,1.7% (1/60 documents)
3,Avoidance Behaviors,Overt or covert actions taken to prevent or es...,merged,safety seeking behavior,1,2,1.666667,60,0.80,It always makes me feel a little better knowin...,1.7% (1/60 documents)
5,Behavioral Change Interventions,Active therapeutic strategies aimed at modifyi...,merged,behavioral intervention,1,1,1.666667,60,1.00,knowing at least a few approaches is quite nec...,1.7% (1/60 documents)
7,Behavioral Change Interventions,Active therapeutic strategies aimed at modifyi...,merged,exposure therapy,1,1,1.666667,60,0.90,I went out eating alone for the first time!,1.7% (1/60 documents)
8,Behavioral Change Interventions,Active therapeutic strategies aimed at modifyi...,merged,exposure and response prevention,1,1,1.666667,60,0.90,I started to push myself out of my comfort zon...,1.7% (1/60 documents)
...,...,...,...,...,...,...,...,...,...,...,...
138,stereotype threat mitigation,The active avoidance of reductive and harmful ...,unplaced,stereotype threat mitigation,1,2,1.666667,60,NaN,actually good representation as opposed to har...,1.7% (1/60 documents)
139,subjective experience of illness,The personal and internal reaction to receivin...,unplaced,subjective experience of illness,1,1,1.666667,60,NaN,What was it like to be diagnosed with autism,1.7% (1/60 documents)
140,substance use as coping,The individual utilizes chemical substances to...,unplaced,substance use as coping,1,1,1.666667,60,NaN,I’m sober for first day in a while like pretty...,1.7% (1/60 documents)
141,therapeutic alliance,The individual is engaging in a collaborative ...,unplaced,therapeutic alliance,1,1,1.666667,60,NaN,My therapist thinks I might be autistic but th...,1.7% (1/60 documents)


Both tables saved to: /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs


## 6. Review and edit the groups

The next cell shows each group name followed by the constructs assigned to it.
This arrangement is called a dictionary in Python. A construct belonging to a
group may also be called a constituent. The grouping file is saved with a name
such as `merged_constructs_groups_<timestamp>.json`, where `<timestamp>` is
replaced by the date and time.

To revise the groups, copy the displayed arrangement into `edited_grouping` in
the following cell, replacing `deepcopy(grouping)`. You can rename a group,
move a construct to another group's list, or remove a construct from the lists.
Keep the quotation marks, commas, and brackets, and list each construct only
once. If you do not want to make changes, leave the code as it is.

This step applies your edits without sending requests to the LLM. The settings
in `apply_merge_edits()` mean:

| Setting | What it means |
|---|---|
| `table` | The results kept in the previous step. Counts and quotes stay with each construct when you change its group. |
| `grouping` | Your revised arrangement of group names and their lists of constructs. |
| `definitions` | Optional new descriptions for groups you rename or change. Otherwise existing descriptions are kept where possible. |
| `total_documents` | Normally carried forward automatically. Editing the groups does not change how many documents were processed. |
| `output_dir` | The folder for the updated results table and grouping files. |
| `save_format` | Choose `"csv"` for a spreadsheet file, `"json"` for a file that keeps lists together, or `"both"`. |

Leaving a construct out of every group removes it from the next steps. Listing
it more than once causes an error. Editing groups does not repeat the original
analysis of your texts.


In [24]:
grouping = {
    name: group["construct_found"].tolist()
    for name, group in threshold_table.groupby("merged_construct", sort=False)
}
print(json.dumps(grouping, indent=2, ensure_ascii=False))


{
  "Avoidance Behaviors": [
    "avoidance behavior",
    "experiential avoidance",
    "maladaptive coping mechanisms"
  ],
  "Behavioral Change Interventions": [
    "behavioral activation"
  ],
  "Cognitive Distortions": [
    "cognitive distortions",
    "catastrophizing",
    "cognitive distortion of labeling"
  ],
  "Cognitive Flexibility and Metacognition": [
    "cognitive flexibility",
    "cognitive appraisal"
  ],
  "Depressive Symptomatology": [
    "depressive affect",
    "depressive symptoms",
    "anhedonia",
    "hopelessness"
  ],
  "Emotional Blunting": [
    "emotional numbness"
  ],
  "Emotional Dysregulation": [
    "emotional dysregulation"
  ],
  "Interpersonal Alienation and Isolation": [
    "interpersonal alienation",
    "social isolation"
  ],
  "Interpersonal Effectiveness and Regulation": [
    "interpersonal effectiveness",
    "collaborative problem solving"
  ],
  "Negative Core Beliefs and Self-Schema": [
    "negative self schema",
    "self stigmat

In [25]:
# Replace this assignment with your edited dictionary, or edit the copy below.
edited_grouping = deepcopy(grouping)

# Examples, using names that actually occur in the printed dictionary:
# edited_grouping["New group name"] = edited_grouping.pop("Old group name")
# edited_grouping["Group name"].remove("constituent to remove")

edited_merged, edited_table = apply_merge_edits(
    table=threshold_table,
    grouping=edited_grouping,
    # definitions={"New group name": "Your revised definition."},
    output_dir=edits_dir,
    save_format="both",
)
display(format_merged_constructs(edited_table, n_quotes=3, seed=seed))


Saved merge output to /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs/edited


,merged_construct,definition,status,construct_found,document_count,instance_count,document_frequency_percent,total_documents,prototypicality,quotes,document_frequency
0,Avoidance Behaviors,Overt or covert actions taken to prevent or es...,merged,avoidance behavior,4,6,6.666667,60,1.00,I’m sitting alone in the cafeteria barely pick...,6.7% (4/60 documents)
1,Avoidance Behaviors,Overt or covert actions taken to prevent or es...,merged,experiential avoidance,7,8,11.666667,60,0.90,What do you do as a means of pushing yourself ...,11.7% (7/60 documents)
2,Avoidance Behaviors,Overt or covert actions taken to prevent or es...,merged,maladaptive coping mechanisms,3,4,5.000000,60,0.70,I get drunk usually every other day now and cr...,5.0% (3/60 documents)
3,Behavioral Change Interventions,Active therapeutic strategies aimed at modifyi...,merged,behavioral activation,4,6,6.666667,60,0.90,Over the years I've attempted lots of exercise...,6.7% (4/60 documents)
4,Cognitive Distortions,Biased or exaggerated thought patterns that re...,merged,cognitive distortions,6,9,10.000000,60,1.00,what's wrong with me that everyone goes out of...,10.0% (6/60 documents)
5,Cognitive Distortions,Biased or exaggerated thought patterns that re...,merged,catastrophizing,12,18,20.000000,60,0.90,now I’m freaking out that it has important inf...,20.0% (12/60 documents)
6,Cognitive Distortions,Biased or exaggerated thought patterns that re...,merged,cognitive distortion of labeling,4,4,6.666667,60,0.80,thinking about all of the ‘bad’ things (not re...,6.7% (4/60 documents)
7,Cognitive Flexibility and Metacognition,The ability to adapt mental frameworks to new ...,merged,cognitive flexibility,2,4,3.333333,60,1.00,I'm trying my best to have an open mind but it...,3.3% (2/60 documents)
8,Cognitive Flexibility and Metacognition,The ability to adapt mental frameworks to new ...,merged,cognitive appraisal,8,12,13.333333,60,0.80,My roommate is really annoying;\nNow that's no...,13.3% (8/60 documents)
9,Depressive Symptomatology,"A cluster of symptoms including low mood, hope...",merged,depressive affect,3,8,5.000000,60,0.95,I feel so low;\nI feel so empty.;\nit fills me...,5.0% (3/60 documents)


Each construct keeps its counts and quotes. The total number of analyzed
documents used to calculate percentages does not change.

Existing definitions are kept where the original group can be identified. If
you create a group that combines constructs from several groups, you may need
to supply a new definition. Review definitions whenever you change membership.

A construct moved to a different group loses its old prototypicality score,
because that score described how well it fit the previous group. Renaming a
group without changing its members keeps the scores. The optional review step
below can update names and definitions, but does not calculate new scores.


## 7. Optionally remove more constructs after your edits

The minimums above have already been applied. This step lets you remove named
constructs or whole groups after reviewing them. The settings below make no
additional cuts unless you change them.

Use `remove_constructs` for individual discovered constructs and
`remove_merged_constructs` for whole groups. Copy names from your results;
capitalization and separators such as underscores are ignored when matching
them. A name not found in the results causes an error.

You can also set additional minimums using `review_min_document_count`,
`review_min_instance_count` (number of supporting quotes), and
`review_min_prototypicality`. A construct must meet every minimum you set;
values exactly at a minimum qualify. A score minimum removes constructs without
scores, including those kept on their own, left unassigned, or moved between
groups. Leave `review_min_prototypicality=None` to avoid removing them for
missing scores. This step sends no requests to the LLM and does not change the
total number of analyzed documents used to calculate percentages.


In [26]:
remove_constructs = []       # Constituent names to remove.
remove_merged_constructs = []  # Whole merged groups to remove.
review_min_document_count = None
review_min_instance_count = None
review_min_prototypicality = None

filtered_merged, filtered_table = filter_merged_constructs(
    table=edited_table,
    remove_constructs=remove_constructs,
    remove_merged_constructs=remove_merged_constructs,
    min_document_count=review_min_document_count,
    min_instance_count=review_min_instance_count,
    min_prototypicality=review_min_prototypicality,
    output_dir=filtered_dir,
    save_format="both",
)
display(format_merged_constructs(filtered_table, n_quotes=3, seed=seed))
print("Constituents remaining:", len(filtered_table))
print("Removed members recorded:", len(filtered_table.attrs.get("removed_constituents", [])))

if filtered_table.empty:
    raise RuntimeError("All constituents were removed. Relax the filters or revise the grouping before building a codebook.")


Saved merge output to /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs/filtered


,merged_construct,definition,status,construct_found,document_count,instance_count,document_frequency_percent,total_documents,prototypicality,quotes,document_frequency
0,Avoidance Behaviors,Overt or covert actions taken to prevent or es...,merged,avoidance behavior,4,6,6.666667,60,1.00,I’m sitting alone in the cafeteria barely pick...,6.7% (4/60 documents)
1,Avoidance Behaviors,Overt or covert actions taken to prevent or es...,merged,experiential avoidance,7,8,11.666667,60,0.90,What do you do as a means of pushing yourself ...,11.7% (7/60 documents)
2,Avoidance Behaviors,Overt or covert actions taken to prevent or es...,merged,maladaptive coping mechanisms,3,4,5.000000,60,0.70,I get drunk usually every other day now and cr...,5.0% (3/60 documents)
3,Behavioral Change Interventions,Active therapeutic strategies aimed at modifyi...,merged,behavioral activation,4,6,6.666667,60,0.90,Over the years I've attempted lots of exercise...,6.7% (4/60 documents)
4,Cognitive Distortions,Biased or exaggerated thought patterns that re...,merged,cognitive distortions,6,9,10.000000,60,1.00,what's wrong with me that everyone goes out of...,10.0% (6/60 documents)
5,Cognitive Distortions,Biased or exaggerated thought patterns that re...,merged,catastrophizing,12,18,20.000000,60,0.90,now I’m freaking out that it has important inf...,20.0% (12/60 documents)
6,Cognitive Distortions,Biased or exaggerated thought patterns that re...,merged,cognitive distortion of labeling,4,4,6.666667,60,0.80,thinking about all of the ‘bad’ things (not re...,6.7% (4/60 documents)
7,Cognitive Flexibility and Metacognition,The ability to adapt mental frameworks to new ...,merged,cognitive flexibility,2,4,3.333333,60,1.00,I'm trying my best to have an open mind but it...,3.3% (2/60 documents)
8,Cognitive Flexibility and Metacognition,The ability to adapt mental frameworks to new ...,merged,cognitive appraisal,8,12,13.333333,60,0.80,My roommate is really annoying;\nNow that's no...,13.3% (8/60 documents)
9,Depressive Symptomatology,"A cluster of symptoms including low mood, hope...",merged,depressive affect,3,8,5.000000,60,0.95,I feel so low;\nI feel so empty.;\nit fills me...,5.0% (3/60 documents)


Constituents remaining: 34
Removed members recorded: 109


## 8. Optional: ask the LLM to review group names and definitions

After your edits, a group's original name or definition may no longer fit its
members. To ask the LLM to review them, set `run_reevaluation=True` in Step 1.
Otherwise this step is skipped. Enabling it sends another request to the LLM
and may use additional credits.

The model is shown the earlier groups, the revised groups, and constructs that
were removed. It can change names and definitions only. It cannot move constructs
between groups or bring back removed ones. Quotes, counts, the total used for
percentages, and all available or missing scores stay unchanged.

The standard instructions are in `REEVALUATE_PROMPT`. You can leave them as
they are. If editing them, keep `{framework}`, `{previous}`, `{removed}`, and
`{current}`, along with the requested answer format, so the notebook can fill
in the information and read the reply. `max_prompt_chars` limits the length of
the instructions and included results before they are sent.


In [27]:
# Print this to inspect the reevaluation instructions without an API call:
# print(REEVALUATE_PROMPT)

if run_reevaluation:
    final_merged, final_table = reevaluate_merged_constructs(
        table=filtered_table,
        config=config,
        framework=framework,
        reevaluation_prompt=REEVALUATE_PROMPT,
        max_prompt_chars=60000,
        output_dir=reevaluation_dir,
        save_format="both",
    )
else:
    final_merged, final_table = filtered_merged, filtered_table
    print("Reevaluation skipped; using the locally reviewed and filtered result.")

display(format_merged_constructs(final_table, n_quotes=3, seed=seed))


Reevaluation skipped; using the locally reviewed and filtered result.


,merged_construct,definition,status,construct_found,document_count,instance_count,document_frequency_percent,total_documents,prototypicality,quotes,document_frequency
0,Avoidance Behaviors,Overt or covert actions taken to prevent or es...,merged,avoidance behavior,4,6,6.666667,60,1.00,I’m sitting alone in the cafeteria barely pick...,6.7% (4/60 documents)
1,Avoidance Behaviors,Overt or covert actions taken to prevent or es...,merged,experiential avoidance,7,8,11.666667,60,0.90,What do you do as a means of pushing yourself ...,11.7% (7/60 documents)
2,Avoidance Behaviors,Overt or covert actions taken to prevent or es...,merged,maladaptive coping mechanisms,3,4,5.000000,60,0.70,I get drunk usually every other day now and cr...,5.0% (3/60 documents)
3,Behavioral Change Interventions,Active therapeutic strategies aimed at modifyi...,merged,behavioral activation,4,6,6.666667,60,0.90,Over the years I've attempted lots of exercise...,6.7% (4/60 documents)
4,Cognitive Distortions,Biased or exaggerated thought patterns that re...,merged,cognitive distortions,6,9,10.000000,60,1.00,what's wrong with me that everyone goes out of...,10.0% (6/60 documents)
5,Cognitive Distortions,Biased or exaggerated thought patterns that re...,merged,catastrophizing,12,18,20.000000,60,0.90,now I’m freaking out that it has important inf...,20.0% (12/60 documents)
6,Cognitive Distortions,Biased or exaggerated thought patterns that re...,merged,cognitive distortion of labeling,4,4,6.666667,60,0.80,thinking about all of the ‘bad’ things (not re...,6.7% (4/60 documents)
7,Cognitive Flexibility and Metacognition,The ability to adapt mental frameworks to new ...,merged,cognitive flexibility,2,4,3.333333,60,1.00,I'm trying my best to have an open mind but it...,3.3% (2/60 documents)
8,Cognitive Flexibility and Metacognition,The ability to adapt mental frameworks to new ...,merged,cognitive appraisal,8,12,13.333333,60,0.80,My roommate is really annoying;\nNow that's no...,13.3% (8/60 documents)
9,Depressive Symptomatology,"A cluster of symptoms including low mood, hope...",merged,depressive affect,3,8,5.000000,60,0.95,I feel so low;\nI feel so empty.;\nit fills me...,5.0% (3/60 documents)


## 9. Save all quotes and a construct summary

This step saves two tables without sending requests to the LLM:

1. **All quotes:** one row for each time a supporting quote was found, with its
   source file, document ID, original construct name, name after spelling
   cleanup, and final group. The same text found in different documents appears
   on separate rows.
2. **Construct summary:** one row per final group, with the columns below.
   There is no extra row combining all groups into an overall result.

The quote table keeps the original quotes even for constructs you removed during
review. Its `retained` column shows whether each construct was kept in the final
groups. Removed constructs have no final group listed. The summary table
contains only the groups you kept after editing and filtering.

| Summary column | What it tells you |
|---|---|
| Construct | The final merged group name. |
| Definition | What that group means. |
| Construct found % (N) | The number of different discovered constructs in this group, and their percentage of all constructs originally found after spelling cleanup. |
| Document count % (N) | The number of different documents containing this group, and their percentage of all successfully analyzed documents. A document with several constructs from the same group counts only once for that group. |
| Instance Count % (N) | The number of supporting quotes for this group, and their percentage of all supporting quotes originally found. Repeated quote text is counted each time it was found. |
| Prototypicality mean (SD) | The mean and standard deviation of the available scores for constructs in this group. Smaller SD means more similar scores. A single score has no SD; missing scores are left out, not treated as zero. |
| Examples | All supporting quote texts for this group, with identical text listed only once. This is not limited to three. |
| Constituents found | The complete list of discovered constructs kept in this group. |

For example, `50.0% (2/4)` means two out of four. Percentages always use the
original totals, even after you remove constructs. A document can contain
constructs from more than one group, so document percentages across groups may
add up to more than 100%.

The next cell uses `build_discovery_tables()` to prepare these tables from the
**original discovery results** (`discoveries`) and your **final reviewed results**
(`final_table`). Keep the same `fuzzy_threshold` setting used earlier so names
match consistently. `output_dir` selects the results folder. `save_format="both"`
saves CSV files for spreadsheets and JSON files that keep lists together. An
additional JSON file records the original totals used to calculate percentages.


In [28]:
all_quotes, construct_summary = build_discovery_tables(
    discoveries=discoveries,
    merged_table=final_table,
    fuzzy_threshold=fuzzy_threshold,
    output_dir=tables_dir,
    save_format="both",
)

# Both tables are saved in full; pandas may shorten the on-screen preview.
with pd.option_context("display.max_colwidth", None):
    display(all_quotes)
    display(construct_summary)
print("Final tables saved to:", tables_dir.resolve())


Saved all original quote occurrences and final construct summary to /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs/final_tables


,filename,doc_id,merged_construct,construct_found,original_construct,description,quote,retained
0,reddit_autism_anxiety_depression.csv,0,Social Anxiety and Evaluation Fears,social anxiety,social anxiety,The individual experiences significant distress and apprehension regarding social interactions and the potential for judgment by others in a workplace setting.,I know my anxiety has made it hard for me to get a job in the past.,True
1,reddit_autism_anxiety_depression.csv,0,Social Anxiety and Evaluation Fears,social anxiety,social anxiety,The individual experiences significant distress and apprehension regarding social interactions and the potential for judgment by others in a workplace setting.,"When i talk to strangers it's one they I'll probably not seem them again, but with coworkers you see them every day.",True
2,reddit_autism_anxiety_depression.csv,0,Negative Core Beliefs and Self-Schema,negative self schema,negative self-schema,The individual holds a core belief that they are fundamentally unappealing or lacking in value compared to others.,I'm not a interesting person,True
3,reddit_autism_anxiety_depression.csv,0,Negative Core Beliefs and Self-Schema,negative self schema,negative self-schema,The individual holds a core belief that they are fundamentally unappealing or lacking in value compared to others.,"i feel like saying watching tv, reading, fishing are all kinda boring and introverted answers.",True
4,reddit_autism_anxiety_depression.csv,0,Avoidance Behaviors,avoidance behavior,avoidance behavior,The individual describes a history of social withdrawal and a current urge to retreat from social demands to mitigate discomfort.,anxiety can make it hard for people to go out and do things.,True
...,...,...,...,...,...,...,...,...
444,reddit_autism_anxiety_depression.csv,59,Interpersonal Alienation and Isolation,social isolation,social isolation,"The individual experiences a profound lack of social support and interpersonal connection, contributing to their distress.",dealing with loneliness,True
445,reddit_autism_anxiety_depression.csv,59,Interpersonal Alienation and Isolation,social isolation,social isolation,"The individual experiences a profound lack of social support and interpersonal connection, contributing to their distress.",when you have 0 friends and been forever alone forever,True
446,reddit_autism_anxiety_depression.csv,59,Interpersonal Alienation and Isolation,social isolation,social isolation,"The individual experiences a profound lack of social support and interpersonal connection, contributing to their distress.",with no close family members,True
447,reddit_autism_anxiety_depression.csv,59,NaN,anxiety,anxiety,The individual identifies persistent feelings of worry or unease as a long-standing component of their mental health struggle.,Ive been dealing with loneliness and anxiety for so long,False


,Construct,Definition,Construct found % (N),Document count % (N),Instance Count % (N),Prototypicality mean (SD),Examples,Constituents found
0,Avoidance Behaviors,Overt or covert actions taken to prevent or escape distressing internal experiences or external situations.,2.1% (3/143),21.7% (13/60),4.0% (18/449),0.87 (0.15),"[anxiety can make it hard for people to go out and do things., as shitty as it sounds i kinda just wanna do my job and leave, I just wish I could move out., I want to know what really make anxiety go away, make myself happy and normal, I NEED TO GO BUT I CANT HELP, I don’t usually step out of my comfort zone and try to peruse something I’m not already good at, What do you do as a means of pushing yourself through the application process?, They can talk shit about the character I created, because it's just a character but when someone is rude to the real me I don't know if I can handle that., everybody has their own problems to I shut up now., I’m sitting alone in the cafeteria barely picking at my food, my anxiety keeps me from going to my classes, inorder to avoid her touching we move back, i couldn't wait to leave that place, I don’t think I’ll come home tonight, I’m to scared to tell my parents because they will go psycho., Sports is a massive part of my life and a coping mechanism., started to smoke and drink because of it, I get drunk usually every other day now and cry my tears dry]","[avoidance behavior, experiential avoidance, maladaptive coping mechanisms]"
1,Behavioral Change Interventions,"Active therapeutic strategies aimed at modifying behavior through reinforcement, exposure, or activation.",0.7% (1/143),6.7% (4/60),1.3% (6/449),0.90 (N/A),"[I went out eating alone for the first time!, I literally have a great job, I volunteer a ton, I get into and out of hobbies, etc..., Over the years I've attempted lots of exercise, races, strength training, I still exercise at a lower level playing racquet sports, baseball, etc.., for the first time in over a year i play as much sport as i could over my Easter break. I was on top of the world., I just sit in my room playing guitar and that's what keeps me occupied.]",[behavioral activation]
2,Cognitive Distortions,Biased or exaggerated thought patterns that reinforce negative emotions and inaccurate perceptions of reality.,2.1% (3/143),33.3% (20/60),6.9% (31/449),0.90 (0.10),"[i feel like saying watching tv, reading, fishing are all kinda boring and introverted answers., idk how to come off as a fun person, I thought moving houses would help but its happened again and i feel so hopless., I feel like I'm going insane, I really cannot deal with living like this., if I survive this thing ;), i know this might be a dumb question, specially among people that really have anxiety, now I’m freaking out that it has important information and that I’ll be left behind in class., I’m basically spiralling right now. Over a goddamn booklet., thinking about all of the ‘bad’ things (not really BAD just things that didn’t go to plan, which stressed me out), I tend to think very pessimistically, what the hell is gonna happen, I really don't know I'm seriously don't know how I'm gonna do this I can't do this, I just get a weird recurring worry that one day I’ll just wake up and realise I’d developed some horrific attraction to children., I just really am freaking out wondering if I made a bad choice to go to school, I should feel happy, but I don’t., Fucking pathetic., I am a fraud and a total failure., I can’t help but feel my life is just a clock ticking down to the moment when the world gathers around me and collectively laughs at how pathetic I am., it was so uncomfortable it almost was life threatening in that balcony, i thought she might even try pushing us, am terrified I won't be asked back because of the personality test, But what if I put him in preprimary and it doesnt help and he needs EIBI next year?, What if I keep him out and put him in EIBI just to not like how 

Final tables saved to: /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs/final_tables


## 10. Build, review, and save a codebook

This step creates your codebook without sending requests to the LLM. Each final
group becomes an entry with a definition and a selection of supporting quotes.
Repeated quote text is listed only once before examples are selected. Examples
are drawn from the group's full set of quotes; there is no guarantee that every
discovered construct in the group will have an example.

Review the definitions and examples before using the codebook. When editing
examples in `codebook_frame`, keep them as lists, such as `["First quote",
"Second quote"]`, rather than one piece of text separated by semicolons.
A quote may itself contain a semicolon, so using semicolons as separators could
split it incorrectly.

The settings in `build_codebook()` mean:

| Setting | What it means |
|---|---|
| `merged` | The final groups you decided to keep, including any revised names and definitions. |
| `name` | A label for this codebook, so you can recognize it later. |
| `version` | Your version number; increase it when making a new reviewed version. |
| `n_examples` | The largest number of example quotes to put into each codebook entry. It does not remove quotes from the discovery results or the tables saved in Step 9. |
| `seed` | Reusing this number with the same results selects the same example quotes. |
| `output_path` | An optional file address for saving immediately. Here you review the codebook first and save it in the following cell. The saved filename includes the date and time. |

Each entry also has space for counterexamples: texts that should **not** be coded
as that construct. These start empty; you can add them during your review.


In [29]:
codebook = build_codebook(
    merged=final_merged,
    name="Discovery tutorial codebook",
    version=1,
    n_examples=n_codebook_examples,
    seed=seed,
)
codebook_frame = codebook_to_frame(codebook)
display(codebook_frame)


,construct,definition,examples,counter_examples
0,Avoidance Behaviors,Overt or covert actions taken to prevent or es...,"[my anxiety keeps me from going to my classes,...",[]
1,Behavioral Change Interventions,Active therapeutic strategies aimed at modifyi...,[Over the years I've attempted lots of exercis...,[]
2,Cognitive Distortions,Biased or exaggerated thought patterns that re...,"[if I survive this thing ;), Sid is called pat...",[]
3,Cognitive Flexibility and Metacognition,The ability to adapt mental frameworks to new ...,[I'm dubious about whether it would be useful....,[]
4,Depressive Symptomatology,"A cluster of symptoms including low mood, hope...","[Anyone else miss the ""childhood magic""?, I do...",[]
5,Emotional Blunting,A state of reduced emotional responsiveness or...,"[Less empty? How do I feel less empty?, But no...",[]
6,Emotional Dysregulation,Difficulty managing and responding to emotiona...,[basically punched and kicked his dad all morn...,[]
7,Interpersonal Alienation and Isolation,A perceived or objective lack of social connec...,"[I feel lonely., I’m alienated and I’m not lik...",[]
8,Interpersonal Effectiveness and Regulation,Skills and processes involved in navigating so...,[i wanna be able to go out maybe make friends ...,[]
9,Negative Core Beliefs and Self-Schema,"Deeply ingrained, globalized negative percepti...","[I feel as though I have nothing to give., i f...",[]


In [30]:
# Make any manual edits to codebook_frame before running this cell.
# Example:
# codebook_frame.at[0, "definition"] = "A clearer definition."
# codebook_frame.at[0, "counter_examples"] = ["A quote that should not count."]

if codebook_frame["definition"].fillna("").str.strip().eq("").any():
    raise ValueError("Fill in missing codebook definitions before saving.")

reviewed_codebook = frame_to_codebook(
    codebook_frame,
    metadata=codebook["metadata"],
)
saved_codebook_path = save_codebook(
    reviewed_codebook,
    codebook_dir / "discovered_codebook.json",
)
print("Use this codebook path in the coding pipeline:", saved_codebook_path.resolve())


Saved codebook to /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs/codebooks/discovered_codebook_20261113_182654_646504.json
Use this codebook path in the coding pipeline: /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs/codebooks/discovered_codebook_20261113_182654_646504.json


## Next: use your codebook in the main tutorial

Continue with the [main tutorial in Google Colab](https://colab.research.google.com/github/childmindresearch/llm_tracker/blob/main/tutorials/tutorial.ipynb)
to use your saved codebook to find constructs in new documents and compare the
results with human coding.
